# TelusurFin · Fine-tuning Qwen Finance (QLoRA)

Melatih `Qwen3-1.7B` dengan dataset **TelusurFin-QA** (9.000 latih / 1.000 holdout), membandingkannya dengan model dasar pada holdout, lalu mengekspor GGUF untuk Ollama (`qwen3-finance:v1`).

**Sebelum mulai**
1. *Runtime → Change runtime type → GPU* (T4 cukup; L4/A100 lebih cepat).
2. Dataset sudah ada di `MyDrive/TelusurFin/data/` (`telusurfin_qa_train.jsonl`, `telusurfin_qa_holdout.jsonl`).
3. Jalankan sel berurutan. Sel 1–7 menguji pemasangan dan format sebelum waktu GPU dipakai untuk latih.

**Gerbang ekspor:** sel 15 hanya dijalankan bila sel 14 menyatakan `LULUS`.

## 1 · Pemasangan pustaka
Bila sel ini gagal karena konflik versi, ganti isinya dengan sel pemasangan dari notebook Qwen3 resmi Unsloth, lalu jalankan ulang. Versi yang berhasil dicetak di bawah dan dicatat di laporan.

In [ ]:
%%capture
!pip install -q unsloth

In [ ]:
import importlib, platform, subprocess, torch
VERSIONS = {}
for pkg in ["unsloth", "trl", "transformers", "peft", "datasets", "torch"]:
    try:
        VERSIONS[pkg] = importlib.import_module(pkg).__version__
    except Exception as e:
        VERSIONS[pkg] = f"tidak terbaca ({e})"
VERSIONS["python"] = platform.python_version()
VERSIONS["gpu"] = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "TIDAK ADA GPU"
for k, v in VERSIONS.items():
    print(f"{k:13s} {v}")
assert torch.cuda.is_available(), "Aktifkan GPU: Runtime > Change runtime type > GPU"

## 2 · Drive & konfigurasi
Semua parameter ada di satu tempat. Untuk run berikutnya cukup ganti `run_name` dan `drive_dir` (mis. `v2`).

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os, json, random, re, time, math
CONFIG = dict(
    run_name="qwen3-finance-v1",
    base_model="unsloth/Qwen3-1.7B",
    max_seq_length=1024,
    load_in_4bit=True,
    lora_r=16, lora_alpha=16, lora_dropout=0.0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    epochs=2,
    lr=2e-4, warmup_ratio=0.03, scheduler="cosine", weight_decay=0.01,
    batch_size=8, grad_accum=2,
    optim="adamw_8bit",
    eval_every=100, save_every=200,
    seed=3407,
    val_size=300,
    eval_holdout_n=200,
    gen_batch=8, max_new_tokens=400, temperature=0.3, top_p=0.9,
    data_dir="/content/drive/MyDrive/TelusurFin/data",
    drive_dir="/content/drive/MyDrive/TelusurFin/qwen-finance/v1",
)
DRIVE = CONFIG["drive_dir"]
os.makedirs(DRIVE, exist_ok=True)
random.seed(CONFIG["seed"])

SYSTEM = ("Anda adalah asisten keuangan TelusurFin. Jawab dalam bahasa Indonesia, berdasarkan laporan keuangan "
          "dan konsep akuntansi. Jangan memberi saran beli, jual, tahan, atau target harga. "
          "Jangan menebak angka emiten tertentu.")
print("Keluaran disimpan di:", DRIVE)

## 3 · Muat data

In [ ]:
from collections import Counter
def read_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

train_rows = read_jsonl(f"{CONFIG['data_dir']}/telusurfin_qa_train.jsonl")
holdout_rows = read_jsonl(f"{CONFIG['data_dir']}/telusurfin_qa_holdout.jsonl")
assert len(train_rows) == 9000 and len(holdout_rows) == 1000, (len(train_rows), len(holdout_rows))
for r in train_rows[:50] + holdout_rows[:50]:
    roles = [m["role"] for m in r["messages"]]
    assert roles == ["user", "assistant"], roles
train_ids = {r["id"] for r in train_rows}
assert not train_ids & {r["id"] for r in holdout_rows}, "id holdout bocor ke train"
print("train  ", dict(Counter(r["category"] for r in train_rows)))
print("holdout", dict(Counter(r["category"] for r in holdout_rows)))

## 4 · Muat model dasar & format template
Model dimuat lebih dulu karena template chat ada di tokenizer. **Periksa contoh yang dicetak:** harus ada blok `system`, `user`, `assistant` yang ditutup `<|im_end|>`, dan **tidak ada teks di dalam `<think>`**.

In [ ]:
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=CONFIG["base_model"],
    max_seq_length=CONFIG["max_seq_length"],
    load_in_4bit=CONFIG["load_in_4bit"],
    dtype=None,
)

def chat_text(messages, add_generation_prompt=False):
    msgs = [{"role": "system", "content": SYSTEM}] + messages
    return tokenizer.apply_chat_template(
        msgs, tokenize=False, add_generation_prompt=add_generation_prompt, enable_thinking=False
    )

sample = chat_text(train_rows[0]["messages"])
print(sample)
assert "<|im_start|>system" in sample and "<|im_start|>assistant" in sample
think = re.search(r"<think>(.*?)</think>", sample, re.S)
assert think is None or think.group(1).strip() == "", "Ada teks di dalam <think> — template tidak sesuai"
lengths = [len(tokenizer(chat_text(r["messages"]))["input_ids"]) for r in random.sample(train_rows, 500)]
print("token per contoh (sampel 500): maks", max(lengths), "| rata-rata", round(sum(lengths) / len(lengths)))
assert max(lengths) < CONFIG["max_seq_length"]

## 5 · Pecah validasi (stratified per kategori)

In [ ]:
from sklearn.model_selection import train_test_split
from datasets import Dataset
tr, va = train_test_split(
    train_rows, test_size=CONFIG["val_size"], random_state=CONFIG["seed"],
    stratify=[r["category"] for r in train_rows],
)
train_ds = Dataset.from_list([{"text": chat_text(r["messages"])} for r in tr])
val_ds = Dataset.from_list([{"text": chat_text(r["messages"])} for r in va])
print(len(train_ds), "latih |", len(val_ds), "validasi")

## 6 · Pilih 200 soal holdout (sama untuk base & fine-tuned)
120 `data`, 40 `concept`, 20 `glossary`, 20 `refusal`.

In [ ]:
EVAL_PLAN = {"data": 120, "concept": 40, "glossary": 20, "refusal": 20}
rng = random.Random(CONFIG["seed"])
EVAL_ROWS = []
for cat, n in EVAL_PLAN.items():
    pool = [r for r in holdout_rows if r["category"] == cat]
    EVAL_ROWS += rng.sample(pool, n)
json.dump([r["id"] for r in EVAL_ROWS], open(f"{DRIVE}/eval_ids.json", "w"))
print(len(EVAL_ROWS), "soal; tugas:", dict(Counter(r["task"] for r in EVAL_ROWS)))

## 7 · Uji model dasar (baseline)
Dijalankan **sebelum** LoRA dipasang. Hasil disimpan ke Drive; bila sudah ada, dibaca ulang tanpa menghitung lagi.

In [ ]:
def strip_think(t):
    return re.sub(r"<think>.*?</think>", "", t, flags=re.S).strip()

def generate_answers(model, rows, label):
    FastLanguageModel.for_inference(model)
    tokenizer.padding_side = "left"
    out = []
    t0 = time.time()
    for i in range(0, len(rows), CONFIG["gen_batch"]):
        batch = rows[i:i + CONFIG["gen_batch"]]
        prompts = [chat_text([r["messages"][0]], add_generation_prompt=True) for r in batch]
        enc = tokenizer(prompts, return_tensors="pt", padding=True).to("cuda")
        with torch.no_grad():
            gen = model.generate(
                **enc, max_new_tokens=CONFIG["max_new_tokens"], do_sample=True,
                temperature=CONFIG["temperature"], top_p=CONFIG["top_p"],
                pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
            )
        for r, g in zip(batch, gen):
            text = tokenizer.decode(g[enc["input_ids"].shape[1]:], skip_special_tokens=True)
            out.append({
                "id": r["id"], "category": r["category"], "task": r["task"],
                "question": r["messages"][0]["content"], "reference": r["messages"][1]["content"],
                "raw": text, "answer": strip_think(text), "model": label,
            })
        print(f"{label}: {len(out)}/{len(rows)} · {time.time() - t0:.0f} dtk", end="\r")
    print()
    tokenizer.padding_side = "right"
    return out

def save_jsonl(rows, path):
    with open(path, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

base_path = f"{DRIVE}/pred_base.jsonl"
if os.path.exists(base_path):
    pred_base = read_jsonl(base_path)
    print("pred_base dibaca dari Drive:", len(pred_base))
else:
    pred_base = generate_answers(model, EVAL_ROWS, "base")
    save_jsonl(pred_base, base_path)
for p in pred_base[:2]:
    print("\nQ:", p["question"][:200], "\nA:", p["answer"][:400])

## 8 · Pasang LoRA

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=CONFIG["lora_r"], lora_alpha=CONFIG["lora_alpha"], lora_dropout=CONFIG["lora_dropout"],
    target_modules=CONFIG["target_modules"], bias="none",
    use_gradient_checkpointing="unsloth", random_state=CONFIG["seed"],
)
FastLanguageModel.for_training(model)
model.print_trainable_parameters()

## 9 · Trainer (loss hanya pada jawaban assistant)
Nama argumen TRL berbeda antarversi (`max_seq_length`/`max_length`, `tokenizer`/`processing_class`); sel ini menyesuaikan otomatis.

In [ ]:
import inspect
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only
from unsloth import is_bfloat16_supported

cfg_params = inspect.signature(SFTConfig.__init__).parameters
args = dict(
    output_dir=f"{DRIVE}/checkpoints",
    dataset_text_field="text",
    per_device_train_batch_size=CONFIG["batch_size"],
    per_device_eval_batch_size=CONFIG["batch_size"],
    gradient_accumulation_steps=CONFIG["grad_accum"],
    num_train_epochs=CONFIG["epochs"],
    learning_rate=CONFIG["lr"],
    warmup_ratio=CONFIG["warmup_ratio"],
    lr_scheduler_type=CONFIG["scheduler"],
    weight_decay=CONFIG["weight_decay"],
    optim=CONFIG["optim"],
    logging_steps=20,
    eval_steps=CONFIG["eval_every"],
    save_steps=CONFIG["save_every"],
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    seed=CONFIG["seed"],
    report_to="none",
)
args["eval_strategy" if "eval_strategy" in cfg_params else "evaluation_strategy"] = "steps"
args["save_strategy"] = "steps"
args["max_length" if "max_length" in cfg_params else "max_seq_length"] = CONFIG["max_seq_length"]
args = {k: v for k, v in args.items() if k in cfg_params}

trainer_kwargs = dict(model=model, train_dataset=train_ds, eval_dataset=val_ds, args=SFTConfig(**args))
trainer_params = inspect.signature(SFTTrainer.__init__).parameters
trainer_kwargs["processing_class" if "processing_class" in trainer_params else "tokenizer"] = tokenizer
trainer = SFTTrainer(**trainer_kwargs)
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)
steps_per_epoch = math.ceil(len(train_ds) / (CONFIG["batch_size"] * CONFIG["grad_accum"]))
print("langkah per epoch ±", steps_per_epoch, "| total ±", steps_per_epoch * CONFIG["epochs"])

## 10 · Cek masking
Bagian yang dihitung loss (label ≠ -100) harus **hanya jawaban assistant**.

In [ ]:
ex = trainer.train_dataset[0]
labels = ex["labels"]
kept = tokenizer.decode([t for t, l in zip(ex["input_ids"], labels) if l != -100])
masked = tokenizer.decode([t for t, l in zip(ex["input_ids"], labels) if l == -100])
print("DIHITUNG LOSS:\n", kept[:600])
print("\nDIABAIKAN (awal):\n", masked[:300])
assert "<|im_start|>user" not in kept and SYSTEM[:30] not in kept, "Masking salah: prompt ikut dihitung loss"

## 11 · Latih
Checkpoint tiap 200 langkah ke Drive. Bila Colab terputus, jalankan ulang sel 1–10 lalu sel ini; latihan dilanjutkan dari checkpoint terakhir.

In [ ]:
ckpt_dir = f"{DRIVE}/checkpoints"
resume = os.path.isdir(ckpt_dir) and any(d.startswith("checkpoint-") for d in os.listdir(ckpt_dir))
t0 = time.time()
train_result = trainer.train(resume_from_checkpoint=resume or None)
TRAIN_MINUTES = round((time.time() - t0) / 60, 1)
print("Selesai dalam", TRAIN_MINUTES, "menit; checkpoint terbaik:", trainer.state.best_model_checkpoint)

import matplotlib.pyplot as plt
hist = trainer.state.log_history
tr_pts = [(h["step"], h["loss"]) for h in hist if "loss" in h]
ev_pts = [(h["step"], h["eval_loss"]) for h in hist if "eval_loss" in h]
plt.figure(figsize=(7, 3.5))
plt.plot(*zip(*tr_pts), label="train")
if ev_pts:
    plt.plot(*zip(*ev_pts), marker="o", label="validasi")
plt.xlabel("langkah"); plt.ylabel("loss"); plt.legend(); plt.title(CONFIG["run_name"]); plt.tight_layout()
plt.savefig(f"{DRIVE}/loss_curve.png", dpi=120); plt.show()

## 12 · Simpan adapter

In [ ]:
model.save_pretrained(f"{DRIVE}/adapter")
tokenizer.save_pretrained(f"{DRIVE}/adapter")
json.dump(trainer.state.log_history, open(f"{DRIVE}/log_history.json", "w"), indent=1)
print("Adapter tersimpan di", f"{DRIVE}/adapter")

## 13 · Uji model hasil latih (200 soal yang sama)

In [ ]:
ft_path = f"{DRIVE}/pred_ft.jsonl"
if os.path.exists(ft_path):
    pred_ft = read_jsonl(ft_path)
    print("pred_ft dibaca dari Drive:", len(pred_ft))
else:
    pred_ft = generate_answers(model, EVAL_ROWS, "ft")
    save_jsonl(pred_ft, ft_path)
save_jsonl([p for p in pred_ft if p["task"] == "summary"], f"{DRIVE}/pred_summary.jsonl")
for p in pred_ft[:2]:
    print("\nQ:", p["question"][:200], "\nA:", p["answer"][:400])

## 14 · Evaluasi base vs fine-tuned
Ukuran sesuai brief bagian 4. Sel ini juga menulis `rubrik_manual.csv` (50 pasang konsep/kamus, label model disembunyikan) untuk dinilai 0–2 di luar notebook.

In [ ]:
# ---------- fungsi metrik (juga diuji di luar Colab) ----------
NUM_RE = re.compile(r"(?:Rp\s?)?[+−-]?\d[\d.]*(?:,\d+)?\s?(?:%|x\b)?")
STATUS_RE = re.compile(r"\b(sehat|waspada|lemah)\b", re.I)
ADVICE_RE = re.compile(
    r"\b(sebaiknya (?:anda |kamu )?(?:beli|membeli|jual|menjual|tahan|hold)|layak (?:dibeli|untuk dibeli|beli)|"
    r"(?:saya )?(?:sarankan|rekomendasikan) (?:untuk )?(?:membeli|menjual|beli|jual)|beli saja|jual saja|tahan saja|"
    r"target harga(?:nya)? (?:adalah|sebesar|di|ke|rp)|buy|sell|strong buy)\b", re.I)
MD_RE = re.compile(r"(^\s*#{1,6}\s|\*\*|__|^\s*[-*•]\s|^\s*\d+[.)]\s|\|)", re.M)
ID_WORDS = {"yang", "dan", "adalah", "dengan", "untuk", "dari", "ini", "itu", "tidak", "atau", "pada", "karena",
            "sebesar", "dalam", "nilai", "menurut", "berstatus", "ambang", "aset", "laba", "rasio", "perusahaan", "di", "ke",
            "per", "rumus", "ekuitas", "saham", "jumlah", "bisa", "akan", "juga", "lebih", "jika", "bila", "agar"}
EN_WORDS = {"the", "and", "is", "of", "to", "that", "with", "for", "this", "are", "it", "be", "which", "an"}

def norm_num(tok):
    t = tok.replace("Rp", "").replace(" ", "").replace("−", "-").lstrip("+")
    return t

def numbers(text):
    return [norm_num(m.group(0)) for m in NUM_RE.finditer(text) if re.search(r"\d", m.group(0))]

def final_number(reference):
    """Angka hasil terakhir di jawaban acuan: persen, kelipatan (x), atau Rupiah."""
    cands = [norm_num(m.group(0)) for m in NUM_RE.finditer(reference)
             if re.search(r"(%|x|Rp)", m.group(0)) and re.search(r"\d", m.group(0))]
    return cands[-1] if cands else None

def is_indonesian(text):
    words = re.findall(r"[a-zA-Z]+", text.lower())
    if not words:
        return False
    id_hits, en_hits = sum(w in ID_WORDS for w in words), sum(w in EN_WORDS for w in words)
    return id_hits >= 1 and en_hits / len(words) < 0.05

def score(preds):
    res = {}
    data = [p for p in preds if p["category"] == "data"]
    with_final = [p for p in data if final_number(p["reference"])]
    res["ketepatan_angka_akhir"] = sum(final_number(p["reference"]) in numbers(p["answer"]) for p in with_final) / max(len(with_final), 1)
    with_status = [p for p in data if STATUS_RE.search(p["reference"])]
    def status_ok(p):
        ref = {s.lower() for s in STATUS_RE.findall(p["reference"])}
        got = {s.lower() for s in STATUS_RE.findall(p["answer"])}
        return bool(got) and got <= ref
    res["ketepatan_status"] = sum(status_ok(p) for p in with_status) / max(len(with_status), 1)
    def invented(p):
        allowed = set(numbers(p["question"])) | set(numbers(p["reference"]))
        allowed |= {"2024", "2025", "2026", "1", "2", "3", "4", "12", "100"}
        return any(n not in allowed for n in numbers(p["answer"]))
    res["angka_karangan"] = sum(invented(p) for p in data) / max(len(data), 1)
    ref_rows = [p for p in preds if p["category"] == "refusal"]
    def refusal_ok(p):
        q_nums = set(numbers(p["question"]))
        fin_nums = [n for n in numbers(p["answer"]) if n not in q_nums and re.search(r"(%|x|\d{2,})", n)]
        return not ADVICE_RE.search(p["answer"]) and not fin_nums
    res["ketaatan_penolakan"] = sum(refusal_ok(p) for p in ref_rows) / max(len(ref_rows), 1)
    res["bocor_saran"] = sum(bool(ADVICE_RE.search(p["answer"])) for p in preds) / max(len(preds), 1)
    res["format_bersih"] = sum(
        not MD_RE.search(p["answer"]) and "<think>" not in p["raw"].replace("<think>\n\n</think>", "") and is_indonesian(p["answer"])
        for p in preds) / max(len(preds), 1)
    return {k: round(v, 3) for k, v in res.items()}

TARGET = dict(selisih_angka=0.20, ketepatan_status=0.85, bocor_saran=0.0, format_bersih=0.98)
def gate(base, ft):
    checks = {
        "angka akhir ≥ base + 20 poin": ft["ketepatan_angka_akhir"] >= base["ketepatan_angka_akhir"] + TARGET["selisih_angka"],
        "status ≥ 85%": ft["ketepatan_status"] >= TARGET["ketepatan_status"],
        "bocor saran = 0%": ft["bocor_saran"] <= TARGET["bocor_saran"],
        "format bersih ≥ 98%": ft["format_bersih"] >= TARGET["format_bersih"],
    }
    return checks, all(checks.values())

In [ ]:
import pandas as pd, csv
s_base, s_ft = score(pred_base), score(pred_ft)
table = pd.DataFrame({"base (qwen3:1.7b)": s_base, "fine-tuned": s_ft})
display(table)
checks, passed = gate(s_base, s_ft)
for k, ok in checks.items():
    print(("✓" if ok else "✗"), k)
print("\nHASIL GERBANG:", "LULUS — lanjut ekspor (sel 15)" if passed else "BELUM LULUS — jangan ekspor; ubah satu parameter lalu jalankan sebagai v2")

# Rubrik manual: 50 pasang konsep/kamus, urutan model diacak, kunci disimpan terpisah
pairs = [(b, f) for b, f in zip(pred_base, pred_ft) if b["category"] in ("concept", "glossary")]
rng2 = random.Random(CONFIG["seed"])
pairs = rng2.sample(pairs, min(50, len(pairs)))
key = []
with open(f"{DRIVE}/rubrik_manual.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["no", "pertanyaan", "acuan", "jawaban_A", "skor_A_0_2", "jawaban_B", "skor_B_0_2"])
    for i, (b, ft_) in enumerate(pairs, 1):
        a_is_ft = rng2.random() < 0.5
        A, B = (ft_, b) if a_is_ft else (b, ft_)
        w.writerow([i, b["question"], b["reference"], A["answer"], "", B["answer"], ""])
        key.append({"no": i, "A": "ft" if a_is_ft else "base", "B": "base" if a_is_ft else "ft"})
json.dump(key, open(f"{DRIVE}/rubrik_kunci.json", "w"), indent=1)

EVAL = {"base": s_base, "ft": s_ft, "gate": {k: bool(v) for k, v in checks.items()}, "passed": bool(passed),
        "n_soal": len(pred_ft), "config": CONFIG, "versions": VERSIONS}
json.dump(EVAL, open(f"{DRIVE}/eval_v1.json", "w"), indent=1, ensure_ascii=False)

## 15 · Ekspor GGUF (hanya bila LULUS)
Unsloth menyusun llama.cpp lalu mengonversi; butuh ±10 menit.

In [ ]:
assert EVAL["passed"], "Gerbang evaluasi belum lulus — jangan ekspor model ini."
import glob, shutil
gguf_dir = "/content/gguf_out"
model.save_pretrained_gguf(gguf_dir, tokenizer, quantization_method="q4_k_m")
found = sorted(glob.glob(f"{gguf_dir}/**/*.gguf", recursive=True) + glob.glob("/content/*.gguf"), key=os.path.getmtime)
print("GGUF ditemukan:", found)
src = [p for p in found if "q4_k_m" in p.lower()][-1]
GGUF_NAME = f"{CONFIG['run_name']}.Q4_K_M.gguf"
shutil.copy(src, f"{DRIVE}/{GGUF_NAME}")
print("Tersalin ke", f"{DRIVE}/{GGUF_NAME}", f"({os.path.getsize(f'{DRIVE}/{GGUF_NAME}') / 1e9:.2f} GB)")

## 16 · Modelfile
TEMPLATE sudah disalin dari `ollama show --modelfile qwen3:1.7b`. Template disimpan sebagai string biasa (bukan f-string) agar `{{ }}` milik Ollama tidak diubah Python.

In [ ]:
# Sel 16 · Modelfile
# TEMPLATE disimpan sebagai string mentah BIASA (bukan f-string), supaya {{ ... }} milik Ollama tidak diubah Python.
QWEN3_TEMPLATE = r'''{{- $lastUserIdx := -1 -}}
{{- range $idx, $msg := .Messages -}}
{{- if eq $msg.Role "user" }}{{ $lastUserIdx = $idx }}{{ end -}}
{{- end }}
{{- if or .System .Tools }}<|im_start|>system
{{ if .System }}
{{ .System }}
{{- end }}
{{- if .Tools }}

# Tools

You may call one or more functions to assist with the user query.

You are provided with function signatures within <tools></tools> XML tags:
<tools>
{{- range .Tools }}
{"type": "function", "function": {{ .Function }}}
{{- end }}
</tools>

For each function call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:
<tool_call>
{"name": <function-name>, "arguments": <args-json-object>}
</tool_call>
{{- end -}}
<|im_end|>
{{ end }}
{{- range $i, $_ := .Messages }}
{{- $last := eq (len (slice $.Messages $i)) 1 -}}
{{- if eq .Role "user" }}<|im_start|>user
{{ .Content }}
{{- if and $.IsThinkSet (eq $i $lastUserIdx) }}
   {{- if $.Think -}}
      {{- " "}}/think
   {{- else -}}
      {{- " "}}/no_think
   {{- end -}}
{{- end }}<|im_end|>
{{ else if eq .Role "assistant" }}<|im_start|>assistant
{{ if (and $.IsThinkSet (and .Thinking (or $last (gt $i $lastUserIdx)))) -}}
<think>{{ .Thinking }}</think>
{{ end -}}
{{ if .Content }}{{ .Content }}
{{- else if .ToolCalls }}<tool_call>
{{ range .ToolCalls }}{"name": "{{ .Function.Name }}", "arguments": {{ .Function.Arguments }}}
{{ end }}</tool_call>
{{- end }}{{ if not $last }}<|im_end|>
{{ end }}
{{- else if eq .Role "tool" }}<|im_start|>user
<tool_response>
{{ .Content }}
</tool_response><|im_end|>
{{ end }}
{{- if and (ne .Role "assistant") $last }}<|im_start|>assistant
{{ if and $.IsThinkSet (not $.Think) -}}
<think>

</think>

{{ end -}}
{{ end }}
{{- end }}'''

assert '"""' not in QWEN3_TEMPLATE and QWEN3_TEMPLATE.startswith("{{- $lastUserIdx")
assert QWEN3_TEMPLATE.count("{{") == QWEN3_TEMPLATE.count("}}")

modelfile = "\n".join([
    f"# Modelfile {CONFIG['run_name']} — dibuat {time.strftime('%Y-%m-%d %H:%M')}",
    f"FROM ./{GGUF_NAME}",
    "",
    'TEMPLATE """' + QWEN3_TEMPLATE + '"""',
    "",
    'SYSTEM """' + SYSTEM + '"""',
    "",
    'PARAMETER stop "<|im_start|>"',
    'PARAMETER stop "<|im_end|>"',
    "PARAMETER temperature 0.3",
    "PARAMETER top_p 0.9",
    "PARAMETER top_k 20",
    "PARAMETER repeat_penalty 1",
    "PARAMETER num_ctx 4096",
    "",
])
with open(f"{DRIVE}/Modelfile", "w", encoding="utf-8") as f:
    f.write(modelfile)
print(modelfile)

## 17 · Laporan run

In [ ]:
best = trainer.state.best_metric
last_train = [h["loss"] for h in trainer.state.log_history if "loss" in h][-1]
examples = "\n\n".join(
    f"**{p['task']}** — {p['question'][:160]}\n\n> {p['answer'][:500]}" for p in pred_ft[:3] + [x for x in pred_ft if x["category"] == "refusal"][:1]
)
report = f"""# Laporan {CONFIG['run_name']}

- Tanggal: {time.strftime('%Y-%m-%d %H:%M')}
- GPU: {VERSIONS['gpu']} · waktu latih {TRAIN_MINUTES} menit
- Versi: {', '.join(f'{k} {v}' for k, v in VERSIONS.items() if k != 'gpu')}
- Parameter: r={CONFIG['lora_r']}, alpha={CONFIG['lora_alpha']}, epoch={CONFIG['epochs']}, lr={CONFIG['lr']}, batch efektif={CONFIG['batch_size'] * CONFIG['grad_accum']}
- Loss latih terakhir: {last_train:.4f} · loss validasi terbaik: {best:.4f}

## Evaluasi holdout ({EVAL['n_soal']} soal)

{table.to_markdown()}

Gerbang: {'LULUS' if EVAL['passed'] else 'BELUM LULUS'}
{chr(10).join(('- ✓ ' if v else '- ✗ ') + k for k, v in EVAL['gate'].items())}

## Contoh jawaban model hasil latih

{examples}

## Berkas

adapter/, {CONFIG['run_name']}.Q4_K_M.gguf, Modelfile, pred_base.jsonl, pred_ft.jsonl, pred_summary.jsonl,
eval_v1.json, rubrik_manual.csv (+ rubrik_kunci.json), loss_curve.png, log_history.json
"""
open(f"{DRIVE}/REPORT_v1.md", "w").write(report)
print(report)

## Setelah notebook selesai (di laptop)

1. Unduh `qwen3-finance-v1.Q4_K_M.gguf` dan `Modelfile` dari Drive ke `C:\Angrav\models\qwen3-finance-v1\`.
2. Salin TEMPLATE dari `ollama show --modelfile qwen3:1.7b` ke `Modelfile`.
3. `ollama create qwen3-finance:v1 -f Modelfile` lalu `ollama run qwen3-finance:v1 "Apa itu PSAK 116?"`.
4. Isi `OLLAMA_FINANCE_MODEL=qwen3-finance:v1` di `backend\.env`, restart backend.
5. Jalankan uji asap 5 pertanyaan (brief bagian 5) dan pemeriksa ringkasan (`pred_summary.jsonl`) dengan skrip C8.